# MSD — folds 0/1/2 × seeds 0/2/20, no augmentation

Reuse every fully verified seed 0/2 result and resume interrupted seed 0/2 work first.
Run missing seed 0/2 combinations, then all missing seed 20 combinations. No seed 1 task is launched.
There are 504 results in the combined target; if seeds 0/2 are complete, only 168 new runs are needed.
Original seed 1 results and recovery remain untouched in the original folder.

**Before switching:** stop the old run cell once and wait for verified recovery uploads. Do not run
this notebook simultaneously with the old notebook. It does not stop or delete the old runtime.
Upload only `msd_seeds0220_resume_noaug_code.zip` to the existing
`My Drive/hippo_msd_3fold_3seed/input/`. Keep the existing data ZIP, original code ZIP and results.
Mount the Google account containing those MSD files. No dataset upload is needed.

Use the same GPU model and software versions recorded in your original results binding. All saved case splits,
10 frozen training cases and 52 validation cases per fold, initialization procedure, losses, calibration, BF16, batch size 1, optimizers,
learning-rate schedules, update counts and 31,976,450,667,090 optimization-FLOP cap remain unchanged.
nnUNet retains native Dice+CE/SGD and 60% targets; the other models retain Dice/AdamW and 10% targets.
**Absolutely no augmentation.** Seed 20 initialization has been exported on CPU and matched against
the completed cluster seed-20 controls. Those cluster results are provenance, not imported Colab runs.

This is a **post-selection exploratory comparison**: seed 1 was replaced after observing scores.
Keep the original seeds-0/1/2 report for transparent reporting. Do not describe this as an unbiased seed sample.


## 1. Mount Drive and stage the pinned code; preserve existing work


In [ ]:
from pathlib import Path
from contextlib import contextmanager, ExitStack
import fcntl, hashlib, json, os, shutil, signal, subprocess, sys, threading, time, zipfile
@contextmanager
def phase(label):
    print('PHASE:', label, flush=True)
    stop=threading.Event(); started=time.monotonic()
    def pulse():
        while not stop.wait(30):
            value=dict(phase=label, elapsed=time.monotonic()-started, status='still waiting; heartbeat is not progress')
            Path('/content/msd_seeds0220_setup_status.json').write_text(json.dumps(value))
            print('HEARTBEAT:', value, flush=True)
    thread=threading.Thread(target=pulse,daemon=True);thread.start()
    try: yield
    finally: stop.set();thread.join()
from google.colab import drive
with phase('mount Drive'): drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/hippo_msd_3fold_3seed')
INPUT=DRIVE_ROOT/'input'
ORIGINAL_RESULTS=DRIVE_ROOT/'results_noaug_v1'
RESULTS=DRIVE_ROOT/'results_seeds0220_noaug_v1'
ORIGINAL_BUNDLE=Path('/content/hippo_msd_bundle')
BUNDLE=Path('/content/hippo_msd_seed20_bundle')
PACKAGE=Path('/content/hippo_msd_seeds0220')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def stage_zip(name, expected, destination):
    local=Path('/content')/name
    with phase('copy '+name+' from Drive to local SSD'):
        shutil.copyfile(INPUT/name,local)
    assert sha256(local)==expected, 'ZIP hash mismatch: '+name
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
        for i in z.infolist():
            p=Path(i.filename)
            assert not p.is_absolute() and '..' not in p.parts and ((i.external_attr>>16)&0o170000)!=0o120000
        z.extractall(destination)
def verify_files(root, manifest):
    for name, expected in json.loads(manifest.read_text()).items():
        assert sha256(root/name)==expected, 'Changed file: '+name
with phase('check original results and ownership'):
    assert (ORIGINAL_RESULTS/'environment_binding.json').is_file(), 'Mount the account containing your existing MSD campaign.'
    for folder in (ORIGINAL_RESULTS,RESULTS):
        assert not (folder/'.msd_controller_lock').exists(), 'Existing controller ownership requires inspection; do not delete it.'
    assert not (RESULTS/'.msd_seeds0220_controller_lock').exists(), 'Existing seeds0220 ownership requires inspection.'
with ExitStack() as locks:
    for root in (ORIGINAL_BUNDLE,BUNDLE):
        (root/'work').mkdir(parents=True,exist_ok=True)
        lock=locks.enter_context((root/'work/sweep.lock').open('a'))
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    for cmdline in Path('/proc').glob('[0-9]*/cmdline'):
        try: words=cmdline.read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError,PermissionError,ProcessLookupError):continue
        for root in (ORIGINAL_BUNDLE,BUNDLE):
            assert not (str(root/'entry.py') in words or (str(root) in words and any(w.endswith(('/worker.py','/phase.py')) for w in words))), 'An MSD worker is still active.'
    with phase('verify or stage original scientific bundle'):
        marker=ORIGINAL_BUNDLE/'bundle_manifest.json'
        if marker.exists():
            assert sha256(marker)=='3b9d0c30f71873b6f834ae3b1fd989bf33e37bef7682fdda094b656daff3620b'
            verify_files(ORIGINAL_BUNDLE,marker)
        else:
            assert not any((ORIGINAL_BUNDLE/'work').glob('*/budget_ledger.json')), 'Preserve and inspect existing work without manifest.'
            stage_zip('msd_replication_code.zip','b0a6dc1a5b94ce8d7ae2506e1583401313cc60fae6735dd4c09f6c73f5e6c51b',ORIGINAL_BUNDLE)
            stage_zip('msd_data_verified.zip','ff045aea37338769009e6322ed55fe08b654681e643ce9cb78782b80e11921f4',ORIGINAL_BUNDLE)
            assert sha256(marker)=='3b9d0c30f71873b6f834ae3b1fd989bf33e37bef7682fdda094b656daff3620b'
            verify_files(ORIGINAL_BUNDLE,marker)
    stage_zip('msd_seeds0220_resume_noaug_code.zip','e034160283755ad6672aaf2bbb7bb1656e275a264a5380b4bbf5c2af4a4dedc2',PACKAGE)
    verify_files(PACKAGE,PACKAGE/'package_manifest.json')
    with phase('stage separate seed20 source; reuse local dataset'):
        marker=BUNDLE/'bundle_manifest.json'
        if marker.exists():
            assert sha256(marker)=='3371089b154f3e1d380d6016c15195f6924d204bef3e118d93cf823779df9e33'
            verify_files(BUNDLE,marker)
        else:
            assert not any((BUNDLE/'work').glob('*/budget_ledger.json')), 'Preserve seed20 work without manifest.'
            shutil.copytree(PACKAGE/'seed20_bundle',BUNDLE,dirs_exist_ok=True)
            (BUNDLE/'data').mkdir(exist_ok=True)
            data=BUNDLE/'data/msd_hippocampus_full.pkl'
            if not data.exists():data.symlink_to(ORIGINAL_BUNDLE/'data/msd_hippocampus_full.pkl')
            verify_files(BUNDLE,marker)
RESULTS.mkdir(parents=True,exist_ok=True)
print('Seeds 0/2 results stay in:',ORIGINAL_RESULTS)
print('Seed20 and combined comparison:',RESULTS)


## 2. Authenticate quota checks and prepare the same training environment


In [ ]:
from google.colab import auth
with phase('authenticate Drive quota API'):auth.authenticate_user()
VENV=Path('/content/hippo_msd_venv');PYTHON=VENV/'bin/python'
def run_logged(args, name):
    path=Path('/content')/('msd_seeds0220_'+name+'.log')
    with path.open('a') as log:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
        def drain():
            for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        interrupted=False
        try:drain();code=p.wait()
        except KeyboardInterrupt:
            interrupted=True
            print('Requesting clean stop; wait for all verified recovery uploads.',flush=True)
            if name in ('run','report'):p.send_signal(signal.SIGINT)
            else:os.killpg(p.pid,signal.SIGINT)
            drain();code=p.wait()
        finally:p.stdout.close()
        if interrupted:raise KeyboardInterrupt('Stopped cleanly; rerun later to resume.')
        if code:raise RuntimeError(f'Exit {code}; inspect {path}. Recovery preserved.')
with phase('prepare training environment'):
    if not PYTHON.exists():run_logged([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup')
    if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
        import urllib.request
        bootstrap=Path('/content/hippo_msd_get_pip.py')
        urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',bootstrap)
        run_logged([PYTHON,bootstrap],'setup')
    run_logged([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',ORIGINAL_BUNDLE/'requirements-colab.txt'],'setup')
COMMON=['--parent-bundle',ORIGINAL_BUNDLE,'--parent-results',ORIGINAL_RESULTS,'--bundle',BUNDLE,'--durable',RESULTS]
def campaign(command,*args):
    run_logged([PYTHON,'-u',PACKAGE/'campaign.py',command,*COMMON,*args],command)
subprocess.run(['nvidia-smi'],check=True)


## 3. Required preflight

The original passed preflight is reused only in its exact environment. Seed20 has a separate
preflight: its four CPU initializations, original loss/calibration/optimizer/evaluation
and exact checkpoint resume are checked. Diagnostics are separate from scientific run accounting.
The seed20 GPU preflight compares the original and accelerated paths on your current GPU.
Its receipt is separate from the retained original preflight.
A failure blocks training. Neither existing environment bindings nor ledgers are reset.


In [ ]:
with phase('verify original environment and run/reuse seed20 preflight'):
    run_logged([PYTHON,'-u',PACKAGE/'preflight.py',*COMMON],'preflight')


## 4. Resume missing seeds 0/2, then run seed 20

Full archive verification runs before any launches. Checked/verified counts and live phases
appear every 30 seconds, with local status independent of Drive. The scheduler fills the CPU-supported
worker target subject to memory/storage/baseline dependencies; no profiling trials are added.
Both phases share one session deadline and launch limit. Every run keeps the original charge-before-update
ledger, recovery uploads, FLOP cap and archive-before-cleanup checks. Seed1 is never launched or removed.
A phase failure prevents the next phase. Original and new STOP files are respected, never cleared.

Interrupt once to stop cleanly and wait for all uploads. Use the same notebook and folders to resume.
Set SESSION_HOURS below the time remaining in your current Colab runtime; 22 hours is only a conservative
target for a newly allocated runtime, not a guarantee. The default 8 hours allows a handover.

The existing speedups remain: GPU data/geometry caches, deferred evaluation readback, unchanged batch-1 evaluation, fewer redundant best-checkpoint writes, local NVMe for new checkpoints, fresh single-read archive verification and maximum eligible worker occupancy without solo profiling or throughput backoff.


In [ ]:
MAX_NEW_RUNS=504  # Maximum missing tasks, never repeats verified results.
MAX_WORKERS=0    # CPU-supported target; actual occupancy depends on resource checks.
SESSION_HOURS=8.0
campaign('run','--max-runs',MAX_NEW_RUNS,'--workers',MAX_WORKERS,'--hours',SESSION_HOURS)


## 5. Combined comparison for seeds0/2/20

This cell re-verifies archives but starts no training. All nine model/condition cells are required
for a final mean: average seeds within each fold, then average the three folds. Gains use the exact
matched model/fold/seed baseline. The original comparison.json is preserved; combined results live
in results_seeds0220_noaug_v1/comparison.json. Seed1 is excluded from this view but remains archived.
Architecture/loss/optimizer confounding and the post-selection seed change must remain explicit.


In [ ]:
campaign('report','--hours',2)
import pandas as pd
result=json.loads((RESULTS/'comparison.json').read_text())
print(f"Verified requested results: {result['completed']}/{result['total']}")
display(pd.DataFrame(result['aggregate']))
display(pd.DataFrame(result['runs']))


Local live status: /content/hippo_msd_bundle/parallel_status and
/content/hippo_msd_seed20_bundle/parallel_status. Notebook logs are /content/msd_seeds0220_*.log.
Original results/recovery remain in results_noaug_v1; new seed20 results/recovery and combined tables
are in results_seeds0220_noaug_v1. Existing completed runs are never migrated or rewritten.
No further dataset upload is needed after a VM reset: rerun setup to stage the existing Drive ZIPs.
Do not delete ownership locks after an unclean termination; inspect recovery/accounting first.
